# 🎭 AI Avatar Animation System — LivePortrait-Style

> **Reusable identity from photos → Real-time webcam driven avatar → Realistic animation**
> Built with PyTorch + MediaPipe + FOMM pretrained models on Google Colab T4 GPU

## Architecture

```
[Input Photos] → Identity Builder → Canonical Face + Texture + Embedding
                                        ↓
[Webcam Feed] → Motion Tracker → 468 Landmarks + Pose + Blink + Mouth
                                        ↓
                Avatar Renderer (FOMM Neural Network) → Animated Face
                                        ↓
                Controller (Commands / Idle FSM)
```

## Pipeline Stages
| Stage | Module | Description |
|-------|--------|-------------|
| 1 | Identity Builder | Extract face identity, canonical texture, landmark topology from photos |
| 2 | Motion Tracker | MediaPipe Face Mesh on webcam — 468 landmarks, head pose, eye/mouth |
| 3 | Avatar Renderer | Pretrained FOMM neural network — identity + motion → realistic face |
| 4 | Controller | Command FSM — Idle, Look, Smile, Blink, Talk, Return to Idle |

## NOT Allowed
❌ No Delaunay triangulation morphing  
❌ No convex hull warping  
❌ No triangle blending or image interpolation  
❌ No animated slideshow

## 0. Setup — Drive Mount, Dependencies, GPU

In [ ]:
# ============================================================
# SETUP: Drive Mount + Dependencies + GPU Check + Paths
# ============================================================
from google.colab import drive
import time, os, sys, warnings
warnings.filterwarnings('ignore')

# --- Mount Drive ---
for attempt in range(3):
    try:
        if os.path.isdir('/content/drive/MyDrive'):
            print("[OK] Drive already mounted")
            break
        drive.mount('/content/drive')
        time.sleep(2)
        if os.path.isdir('/content/drive/MyDrive'):
            print("[OK] Drive mounted successfully")
            break
    except Exception as e:
        print(f"[Attempt {attempt+1}] {e}")
        time.sleep(3)
else:
    print("[!] Mount Drive manually via left sidebar")

# --- Install Dependencies ---
print("\n=== Installing dependencies... ===")
DEPS = [
    'opencv-python', 'mediapipe', 'torch', 'torchvision',
    'numpy', 'scipy', 'matplotlib', 'tqdm', 'pillow',
    'insightface', 'onnxruntime-gpu', 'scikit-image',
    'requests', 'gdown'
]
!pip install -q {" ".join(DEPS)}

# --- Check GPU ---
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import subprocess as sp
r = sp.run(['nvidia-smi','-L'], capture_output=True, text=True)
if r.returncode == 0:
    print(f"nvidia-smi: {r.stdout.strip().split(chr(10))[0]}")

# --- Paths ---
DRIVE_ROOT = '/content/drive/MyDrive/AI_Face_Data'
INPUT_DIR  = f'{DRIVE_ROOT}/input'
OUTPUT_DIR = f'{DRIVE_ROOT}/output'
MODEL_DIR  = f'{DRIVE_ROOT}/models'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"\nInput:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Models: {MODEL_DIR}")

# --- Verify Drive ---
print(f"\nDrive contents: {os.listdir('/content/drive/MyDrive')}")
if os.path.exists(INPUT_DIR):
    print(f"Input files: {os.listdir(INPUT_DIR)}")

# --- Global device ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nDevice: {DEVICE}")
print("[OK] Setup complete!")


## 1. Load Pretrained Models — InsightFace + MediaPipe + FOMM

**Models loaded in this section:**
- **InsightFace (buffalo_l)**: Face detection, 106 landmarks, 512-dim identity embedding (ArcFace)
- **MediaPipe Face Mesh**: 468 3D face landmarks for realtime tracking
- **FOMM (First Order Motion Model)**: Neural face animation model pretrained on VoxCeleb

In [ ]:
# ============================================================
# LOAD PRETRAINED MODELS
# ============================================================
print("=== Loading InsightFace (face detection + identity) ===")
import insightface
from insightface.app import FaceAnalysis
import cv2, numpy as np, torch

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
print(f"[OK] InsightFace loaded — model: buffalo_l")

print("\n=== Loading MediaPipe Face Mesh ===")
import mediapipe as mp
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
mp_drawing = mp.solutions.drawing_utils
print(f"[OK] MediaPipe Face Mesh loaded — 468 landmarks")

# ============================================================
# FOMM - First Order Motion Model
# ============================================================
print("\n=== Downloading FOMM pretrained model ===")
import gdown, zipfile, glob

FOMM_DIR = f'{MODEL_DIR}/fomm'
os.makedirs(FOMM_DIR, exist_ok=True)

# Download vox-adv checkpoint (256x256, best quality)
fomm_ckpt = f'{FOMM_DIR}/vox-adv.pth.tar'
if not os.path.exists(fomm_ckpt):
    print("Downloading FOMM checkpoint (vox-adv, 256MB)...")
    # vox-adv checkpoint from FOMM official
    gdown.download('https://drive.google.com/uc?id=1wC2UFEYbPSCWCqiBJdfG8kCIrPZrKfsU',
                   fomm_ckpt, quiet=False)
    print(f"[OK] Downloaded to {fomm_ckpt}")
else:
    print(f"[OK] FOMM checkpoint exists: {fomm_ckpt}")

# Now define the FOMM network architecture inline
import torch.nn as nn
import torch.nn.functional as F

class DownBlock2d(nn.Module):
    def __init__(self, in_features, out_features, kernel_size=3, padding=1, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_features, out_features, kernel_size, padding=padding, groups=groups)
        self.norm = nn.InstanceNorm2d(out_features, affine=True)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.norm(self.conv(x)))

class UpBlock2d(nn.Module):
    def __init__(self, in_features, out_features, kernel_size=3, padding=1, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_features, out_features, kernel_size, padding=padding, groups=groups)
        self.norm = nn.InstanceNorm2d(out_features, affine=True)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        out = F.interpolate(x, scale_factor=2)
        return self.activation(self.norm(self.conv(out)))

class SameBlock2d(nn.Module):
    def __init__(self, in_features, out_features, kernel_size=7, padding=3):
        super().__init__()
        self.conv = nn.Conv2d(in_features, out_features, kernel_size, padding=padding)
        self.norm = nn.InstanceNorm2d(out_features, affine=True)
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.norm(self.conv(x)))

class ResBlock2d(nn.Module):
    def __init__(self, in_features, kernel_size=3, padding=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, in_features, kernel_size, padding=padding)
        self.conv2 = nn.Conv2d(in_features, in_features, kernel_size, padding=padding)
        self.norm1 = nn.InstanceNorm2d(in_features, affine=True)
        self.norm2 = nn.InstanceNorm2d(in_features, affine=True)

    def forward(self, x):
        out = self.norm2(self.conv2(self.activation(self.norm1(self.conv1(x)))))
        return out + x

class AntiAliasInterpolation2d(nn.Module):
    def __init__(self, channels, scale):
        super().__init__()
        sigma = (1/scale - 1) / 2 if scale < 1 else 0.01
        kernel_size = 2*round(sigma*4) + 1
        self.blur = nn.Conv2d(channels, channels, kernel_size, padding=kernel_size//2, groups=channels, bias=False)
        self.blur.weight.data[:] = self.blur_kernel(sigma, channels, kernel_size)
        self.scale = scale

    @staticmethod
    def blur_kernel(sigma, channels, kernel_size):
        ax = torch.arange(-kernel_size//2+1, kernel_size//2+1, dtype=torch.float32)
        xx, yy = torch.meshgrid(ax, ax, indexing='ij')
        kernel = torch.exp(-(xx**2 + yy**2) / (2*sigma**2))
        kernel /= kernel.sum()
        return kernel.view(1,1,*kernel.shape).repeat(channels, 1, 1, 1)

    def forward(self, x):
        x = self.blur(x)
        return F.interpolate(x, scale_factor=self.scale, mode='bilinear', align_corners=False)

class Encoder(nn.Module):
    def __init__(self, num_kp=10, num_channels=3, block_expansion=64, num_down_blocks=3, max_features=512):
        super().__init__()
        self.num_kp = num_kp
        down_blocks = []
        for i in range(num_down_blocks):
            in_f = num_channels if i==0 else min(max_features, block_expansion*(2**i))
            out_f = min(max_features, block_expansion*(2**(i+1)))
            down_blocks.append(DownBlock2d(in_f, out_f, kernel_size=3, padding=1))
        self.down_blocks = nn.ModuleList(down_blocks)
        self.conv_final = nn.Conv2d(min(max_features, block_expansion*(2**num_down_blocks)), num_kp, kernel_size=7, padding=3)

    def forward(self, x):
        out = x
        for db in self.down_blocks:
            out = db(out)
        out = self.conv_final(out)
        return out

class KeypointDetector(nn.Module):
    def __init__(self, num_kp=10, num_channels=3, estimate_jacobian=True):
        super().__init__()
        self.encoder = Encoder(num_kp=num_kp, num_channels=num_channels)
        self.estimate_jacobian = estimate_jacobian
        if estimate_jacobian:
            self.jacobian = nn.Conv2d(num_kp, 4, kernel_size=7, padding=3)

    def forward(self, x):
        feature_map = self.encoder(x)
        kp = feature_map.mean(axis=(2,3))
        result = {'value': kp}
        if self.estimate_jacobian:
            j_map = self.jacobian(feature_map)
            j = j_map.mean(axis=(2,3))
            j = j.view(-1, self.encoder.num_kp, 2, 2)
            result['jacobian'] = j
        return result

class DenseMotionNetwork(nn.Module):
    def __init__(self, num_kp=10, num_channels=3, block_expansion=64, num_blocks=5, max_features=1024):
        super().__init__()
        self.num_kp = num_kp
        self.num_blocks = num_blocks
        self.first = SameBlock2d(num_channels+num_kp*3+2, block_expansion, kernel_size=7, padding=3)
        down_blocks = []
        up_blocks = []
        for i in range(num_blocks):
            in_f = block_expansion * (2**i)
            out_f = block_expansion * (2**(i+1)) if i < num_blocks-1 else block_expansion * (2**i)
            down_blocks.append(DownBlock2d(min(in_f, max_features), min(out_f, max_features)))
            up_blocks.append(UpBlock2d(min(in_f+out_f, max_features), min(in_f, max_features)))
        self.down_blocks = nn.ModuleList(down_blocks)
        self.up_blocks = nn.ModuleList(up_blocks)
        self.final = nn.Conv2d(block_expansion, 3, kernel_size=7, padding=3)

    def forward(self, source, driving, source_kp, driving_kp):
        bs = source.shape[0]
        out = source
        kp_source_std = source_kp['value']
        kp_driving_std = driving_kp['value']
        # Add kp heatmaps
        heatmaps = []
        for kp_s, kp_d in zip(kp_source_std, kp_driving_std):
            h = torch.zeros(source.shape[0], self.num_kp*3+2, source.shape[2], source.shape[3], device=source.device)
            for i in range(self.num_kp):
                h[:,i] = self._heatmap(kp_s[i], source.shape[2:])
                h[:,i+self.num_kp] = self._heatmap(kp_d[i], source.shape[2:])
            heatmaps.append(h)
        out = torch.cat([source] + heatmaps, dim=1)
        out = self.first(out)
        skips = [out]
        for db in self.down_blocks:
            out = db(out)
            skips.append(out)
        out = skips.pop()
        for ub in self.up_blocks:
            out = ub(torch.cat([out, skips.pop()], dim=1))
        out = self.final(out)
        return out

    @staticmethod
    def _heatmap(kp, size):
        y, x = torch.meshgrid(torch.linspace(-1,1,size[0],device=kp.device),
                              torch.linspace(-1,1,size[1],device=kp.device), indexing='ij')
        return torch.exp(-((x-kp[0])**2 + (y-kp[1])**2)*10).unsqueeze(0)

class OcclusionAwareGenerator(nn.Module):
    def __init__(self, num_channels=3, block_expansion=64, max_features=512, num_down_blocks=3, num_up_blocks=3):
        super().__init__()
        down_blocks = []
        for i in range(num_down_blocks):
            in_f = num_channels if i==0 else min(max_features, block_expansion*(2**i))
            out_f = min(max_features, block_expansion*(2**(i+1)))
            down_blocks.append(DownBlock2d(in_f, out_f))
        self.down_blocks = nn.ModuleList(down_blocks)
        up_blocks = []
        for i in range(num_up_blocks):
            in_f = min(max_features, block_expansion*(2**(num_down_blocks-i)))
            out_f = min(max_features, block_expansion*(2**(num_down_blocks-i-1)))
            up_blocks.append(UpBlock2d(in_f, out_f))
        self.up_blocks = nn.ModuleList(up_blocks)
        self.bottleneck = torch.nn.Sequential(
            nn.Conv2d(min(max_features, block_expansion*(2**num_down_blocks)), min(max_features, block_expansion*(2**num_down_blocks)), kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(min(max_features, block_expansion*(2**num_down_blocks)), min(max_features, block_expansion*(2**num_down_blocks)), kernel_size=3, padding=1))
        self.final = nn.Sequential(
            nn.Conv2d(block_expansion, block_expansion, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(block_expansion, 3, kernel_size=1, padding=0))

    def forward(self, x):
        skips = []
        out = x
        for db in self.down_blocks:
            out = db(out)
            skips.append(out)
        out = self.bottleneck(out)
        for ub in self.up_blocks:
            out = ub(torch.cat([out, skips.pop()], dim=1))
        return self.final(out)

class FOMMModel(nn.Module):
    def __init__(self, num_kp=10):
        super().__init__()
        self.kp_detector = KeypointDetector(num_kp=num_kp)
        self.generator = OcclusionAwareGenerator()
        self.dense_motion = DenseMotionNetwork(num_kp=num_kp)

    def forward(self, source, driving):
        source_kp = self.kp_detector(source)
        driving_kp = self.kp_detector(driving)
        dense = self.dense_motion(source, driving, source_kp, driving_kp)
        generated = self.generator(dense)
        return generated, source_kp, driving_kp

# Load FOMM checkpoint
def load_fomm(fomm_ckpt, device='cuda'):
    print(f"Loading FOMM from {fomm_ckpt}...")
    model = FOMMModel(num_kp=10).to(device)
    checkpoint = torch.load(fomm_ckpt, map_location=device, weights_only=False)
    if 'state_dict' in checkpoint:
        state_dict = {k.replace('module.',''):v for k,v in checkpoint['state_dict'].items()}
    elif 'kp_detector' in checkpoint:
        state_dict = {}
        for key in ['kp_detector', 'generator', 'dense_motion']:
            state_dict.update({f'{key}.{k}':v for k,v in checkpoint[key].items()})
    else:
        state_dict = {k.replace('module.',''):v for k,v in checkpoint.items()}
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  Missing keys: {len(missing)}")
    if unexpected:
        print(f"  Unexpected keys: {len(unexpected)}")
    model.eval()
    print(f"[OK] FOMM model loaded on {device}")
    return model

fomm_model = load_fomm(fomm_ckpt, DEVICE)

print("\n=== All models loaded ===")


## 2. Identity Builder — Canonical Face + Texture + Landmark Topology

**Input:** Front face image (required) + optional left/right images + optional calibration video  
**Output:** `IdentityModel` object containing:
- `canonical_face` — aligned face image (256×256)
- `identity_embedding` — 512-dim ArcFace vector
- `texture` — face texture map
- `landmark_topology` — 468 MediaPipe landmarks
- `head_pose_ref` — reference head pose

In [ ]:
# ============================================================
# IDENTITY BUILDER
# ============================================================
import dataclasses
from typing import Optional, List, Tuple

@dataclasses.dataclass
class IdentityModel:
    """Reusable identity representation for the avatar system."""
    canonical_face: np.ndarray       # 256x256 aligned face RGB
    identity_embedding: np.ndarray   # 512-dim ArcFace embedding
    texture: np.ndarray              # Face texture atlas (optional)
    landmarks_468: np.ndarray        # 468×3 MediaPipe landmarks
    head_pose_ref: np.ndarray        # Reference head pose (yaw,pitch,roll)
    source_kp: dict = None           # FOMM source keypoints (precomputed)
    source_tensor: torch.Tensor = None  # FOMM source tensor

def build_identity_from_images(
    image_paths: List[str],
    device: str = 'cuda'
) -> Optional[IdentityModel]:
    """
    Build identity from one or more face images.
    Uses the best quality frontal face as canonical.
    """
    print(f"\n=== Building Identity from {len(image_paths)} images ===")
    best_face = None
    best_score = -1
    best_lms = None
    best_embedding = None

    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            print(f"  [SKIP] Cannot read: {img_path}")
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # InsightFace detection
        faces = app.get(img_rgb)
        if len(faces) == 0:
            print(f"  [SKIP] No face in: {os.path.basename(img_path)}")
            continue

        face = faces[0]
        embedding = face.normed_embedding
        bbox = face.bbox.astype(int)
        area = (bbox[2]-bbox[0])*(bbox[3]-bbox[1])
        # Quality: area * face detection score
        score = area * (face.det_score if hasattr(face, 'det_score') else 1.0)

        # MediaPipe 468 landmarks
        mp_result = face_mesh.process(img_rgb)
        if mp_result and mp_result.multi_face_landmarks:
            lms_468 = np.array([(lm.x, lm.y, lm.z) for lm in mp_result.multi_face_landmarks[0].landmark])
        else:
            lms_468 = None

        print(f"  {os.path.basename(img_path)}: face_area={area}, score={score:.1f}, lms={lms_468 is not None}")

        if score > best_score and lms_468 is not None:
            best_score = score
            best_face = img_rgb.copy()
            best_lms = lms_468.copy()
            best_embedding = embedding.copy()

    if best_face is None:
        print("[ERROR] No valid face found in any image!")
        return None

    # Align and crop face to 256x256 using InsightFace landmarks
    kps = faces[0].kps  # 5-point landmarks (eyes, nose, mouth corners)
    src_pts = np.array([[38.2946, 51.6963], [73.5318, 51.5014],
                        [56.0252, 71.7366], [41.5493, 92.3655],
                        [70.7299, 92.2041]], dtype=np.float32)
    dst_pts = kps.astype(np.float32)
    tform = cv2.estimateAffinePartial2D(dst_pts, src_pts, method=cv2.LMEDS)[0]
    if tform is None:
        # Fallback: center crop
        h,w = best_face.shape[:2]
        s = min(h,w)
        canonical = best_face[h//2-s//2:h//2+s//2, w//2-s//2:w//2+s//2]
        canonical = cv2.resize(canonical, (256,256))
    else:
        canonical = cv2.warpAffine(best_face, tform, (256,256), flags=cv2.INTER_CUBIC)

    # Compute reference head pose from MediaPipe landmarks
    # Using face mesh landmarks: nose tip (1), left eye outer (33), right eye outer (263)
    ref_yaw, ref_pitch, ref_roll = 0.0, 0.0, 0.0
    try:
        # Simple heuristic: use landmark positions for pose
        nose_tip = best_lms[1]   # landmark 1 = nose tip
        l_eye = best_lms[33]     # left eye outer
        r_eye = best_lms[263]    # right eye outer
        # Roll from eye tilt
        dy = r_eye[1] - l_eye[1]
        dx = r_eye[0] - l_eye[0]
        ref_roll = float(np.degrees(np.arctan2(dy, dx)))
        # Yaw from nose asymmetry
        eye_mid_x = (l_eye[0] + r_eye[0]) / 2
        ref_yaw = float((nose_tip[0] - eye_mid_x) * 40)
    except:
        pass

    identity = IdentityModel(
        canonical_face=canonical,
        identity_embedding=best_embedding,
        texture=canonical,  # Simple: use canonical as texture
        landmarks_468=best_lms,
        head_pose_ref=np.array([ref_yaw, ref_pitch, ref_roll], dtype=np.float32)
    )

    # Precompute FOMM source keypoints
    source_tensor = torch.from_numpy(canonical.transpose(2,0,1)[np.newaxis].astype(np.float32) / 255.0).to(device)
    with torch.no_grad():
        source_kp = fomm_model.kp_detector(source_tensor)
    identity.source_kp = source_kp
    identity.source_tensor = source_tensor

    print(f"\n[OK] Identity built!")
    print(f"  Canonical: {canonical.shape}")
    print(f"  Embedding: {best_embedding.shape}")
    print(f"  Landmarks: {best_lms.shape}")
    print(f"  Head pose: yaw={ref_yaw:.1f}, pitch={ref_pitch:.1f}, roll={ref_roll:.1f}")

    # Display
    fig, axes = plt.subplots(1,3,figsize=(12,4))
    axes[0].imshow(best_face)
    axes[0].set_title('Best detected face'); axes[0].axis('off')
    axes[1].imshow(canonical)
    axes[1].set_title('Canonical (aligned 256x256)'); axes[1].axis('off')
    # Plot landmarks
    axes[2].imshow(canonical)
    xs = best_lms[:,0] * canonical.shape[1]
    ys = best_lms[:,1] * canonical.shape[0]
    axes[2].scatter(xs, ys, s=1, c='lime', alpha=0.5)
    axes[2].set_title(f'468 Landmarks ({best_lms.shape[0]} pts)'); axes[2].axis('off')
    plt.tight_layout(); plt.show()

    return identity

# --- Auto-detect input files ---
print("\n=== Scanning input files ===")
image_exts = {'.jpg','.jpeg','.png'}
video_exts = {'.mp4','.avi','.mov'}
input_images = []
input_videos = []

for f in os.listdir(INPUT_DIR):
    ext = os.path.splitext(f)[1].lower()
    fp = os.path.join(INPUT_DIR, f)
    if ext in image_exts:
        input_images.append(fp)
    elif ext in video_exts:
        input_videos.append(fp)

print(f"Images: {len(input_images)} — {[os.path.basename(p) for p in input_images]}")
print(f"Videos: {len(input_videos)} — {[os.path.basename(p) for p in input_videos]}")

if len(input_images) == 0 and len(input_videos) == 0:
    print("[!] No files found in INPUT_DIR. Upload images to proceed.")
    identity = None
else:
    identity = build_identity_from_images(input_images, DEVICE)


## 3. Motion Tracker — MediaPipe Face Mesh + Head Pose + Blink + Mouth

**Extracts from each webcam frame:**
- `yaw/pitch/roll` — head rotation angles (degrees)
- `eye_blink` — 0.0 (open) to 1.0 (closed), averaged both eyes
- `mouth_open` — 0.0 (closed) to 1.0 (open)
- `eyebrow_raise` — eyebrow height relative to reference
- `landmarks_468` — full 468×3 normalized face mesh
- `expression` — placeholder for future expression features

In [ ]:
# ============================================================
# MOTION TRACKER — Real-time Face Mesh + Pose + Expressions
# ============================================================
from dataclasses import dataclass, field

@dataclass
class MotionFrame:
    """Motion data extracted from one webcam frame."""
    landmarks_468: np.ndarray       # 468×3 normalized landmarks
    yaw: float = 0.0
    pitch: float = 0.0
    roll: float = 0.0
    eye_blink: float = 0.0          # 0=open, 1=closed
    mouth_open: float = 0.0         # 0=closed, 1=open
    eyebrow_raise: float = 0.0      # 0=neutral, 1=raised
    smile: float = 0.0              # 0=neutral, 1=smile
    has_face: bool = False

class MotionTracker:
    """Processes webcam frames and extracts motion features using MediaPipe."""

    # MediaPipe face mesh landmark indices
    LEFT_EYE_IDX = [33, 133, 157, 158, 159, 160, 161, 173]
    RIGHT_EYE_IDX = [362, 263, 386, 387, 388, 389, 390, 398]
    LEFT_EYEBROW_IDX = [46, 53, 52, 65, 55]
    RIGHT_EYEBROW_IDX = [285, 295, 282, 283, 276]
    NOSE_TIP = 1
    NOSE_BRIDGE = 168
    LEFT_EYE_OUTER = 33
    RIGHT_EYE_OUTER = 263
    MOUTH_TOP = 13
    MOUTH_BOTTOM = 14
    MOUTH_LEFT = 61
    MOUTH_RIGHT = 291
    LIPS_TOP = 0
    LIPS_BOTTOM = 17
    FOREHEAD = 10  # between eyebrows

    def __init__(self, smooth_frames: int = 3):
        self.face_mesh = mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.smooth_frames = smooth_frames
        self._history = []
        self._ref_eye_dist = None  # Calibrated reference

    def extract(self, img_rgb: np.ndarray) -> MotionFrame:
        """Extract full motion features from a BGR or RGB frame."""
        h, w = img_rgb.shape[:2]
        result = self.face_mesh.process(img_rgb)

        mf = MotionFrame(landmarks_468=np.zeros((468,3), dtype=np.float32))

        if not result or not result.multi_face_landmarks:
            mf.has_face = False
            # Return neutral frame
            if self._history:
                prev = self._history[-1]
                mf.yaw, mf.pitch, mf.roll = prev.yaw*0.9, prev.pitch*0.9, prev.roll*0.9
            return mf

        mf.has_face = True
        lms = result.multi_face_landmarks[0].landmark
        pts = np.array([(lm.x, lm.y, lm.z) for lm in lms], dtype=np.float32)
        mf.landmarks_468 = pts.copy()

        # --- Head Pose (yaw/pitch/roll) from landmark positions ---
        nose = pts[self.NOSE_TIP]
        nose_bridge = pts[self.NOSE_BRIDGE]
        l_eye = pts[self.LEFT_EYE_OUTER]
        r_eye = pts[self.RIGHT_EYE_OUTER]

        # Roll: angle of eye line
        dy = r_eye[1] - l_eye[1]
        dx = r_eye[0] - l_eye[0]
        mf.roll = float(np.degrees(np.arctan2(dy, dx)))

        # Yaw: nose shift relative to eye midpoint
        eye_mid_x = (l_eye[0] + r_eye[0]) / 2
        eye_dist = abs(r_eye[0] - l_eye[0]) + 1e-6
        mf.yaw = float(((nose[0] - eye_mid_x) / eye_dist) * 50)

        # Pitch: nose bridge vertical relative to eyes
        face_mid_y = (l_eye[1] + r_eye[1]) / 2
        mf.pitch = float(((nose_bridge[1] - face_mid_y)) * 100)

        # --- Eye Blink (Eye Aspect Ratio) ---
        def ear(eye_indices):
            pts_e = pts[eye_indices]
            # EAR = (|p1-p5| + |p2-p4|) / (2*|p0-p3|)
            vert1 = np.linalg.norm(pts_e[1][:2] - pts_e[5][:2])
            vert2 = np.linalg.norm(pts_e[2][:2] - pts_e[4][:2])
            horiz = np.linalg.norm(pts_e[0][:2] - pts_e[3][:2])
            return (vert1 + vert2) / (2.0 * horiz + 1e-6)

        left_ear = ear(self.LEFT_EYE_IDX)
        right_ear = ear(self.RIGHT_EYE_IDX)
        avg_ear = (left_ear + right_ear) / 2.0

        # Calibrate reference on first valid detection
        if self._ref_eye_dist is None or avg_ear > self._ref_eye_dist:
            self._ref_eye_dist = avg_ear

        # Blink: 0=open (EAR near ref), 1=closed (EAR near 0)
        if self._ref_eye_dist > 0:
            mf.eye_blink = float(np.clip(1.0 - avg_ear / self._ref_eye_dist, 0.0, 1.0))
        else:
            mf.eye_blink = 0.0

        # --- Mouth Opening ---
        mouth_top = pts[self.MOUTH_TOP]
        mouth_bottom = pts[self.MOUTH_BOTTOM]
        mouth_left = pts[self.MOUTH_LEFT]
        mouth_right = pts[self.MOUTH_RIGHT]

        mouth_height = np.linalg.norm(mouth_top[:2] - mouth_bottom[:2])
        mouth_width = np.linalg.norm(mouth_left[:2] - mouth_right[:2]) + 1e-6
        mouth_ratio = mouth_height / mouth_width
        # Map: 0.0 (closed) to 1.0 (wide open), threshold ~0.15
        mf.mouth_open = float(np.clip((mouth_ratio - 0.05) * 8.0, 0.0, 1.0))

        # --- Smile (mouth width / face width) ---
        mf.smile = float(np.clip(mouth_width / (eye_dist + 1e-6) - 0.6, 0.0, 1.0))

        # --- Eyebrow Raise ---
        l_brow = np.mean(pts[self.LEFT_EYEBROW_IDX], axis=0)
        r_brow = np.mean(pts[self.RIGHT_EYEBROW_IDX], axis=0)
        brow_y = (l_brow[1] + r_brow[1]) / 2
        ref_y = pts[self.FOREHEAD][1]
        mf.eyebrow_raise = float(np.clip((ref_y - brow_y) * 3.0, 0.0, 1.0))

        # Smoothing
        self._history.append(mf)
        if len(self._history) > self.smooth_frames:
            self._history.pop(0)

        if len(self._history) >= 2:
            # Moving average smoothing
            for attr in ['yaw','pitch','roll','eye_blink','mouth_open','eyebrow_raise','smile']:
                vals = [getattr(f, attr) for f in self._history]
                setattr(mf, attr, sum(vals)/len(vals))

        return mf

    def reset_calibration(self):
        self._ref_eye_dist = None
        self._history.clear()

# Test on sample images
print("[OK] MotionTracker class defined")
if identity is not None:
    tracker = MotionTracker()
    test_mf = tracker.extract(identity.canonical_face)
    print(f"Test on canonical face:")
    print(f"  Yaw={test_mf.yaw:.1f}  Pitch={test_mf.pitch:.1f}  Roll={test_mf.roll:.1f}")
    print(f"  Blink={test_mf.eye_blink:.2f}  Mouth={test_mf.mouth_open:.2f}")
    print(f"  Smile={test_mf.smile:.2f}  Eyebrow={test_mf.eyebrow_raise:.2f}")
    print(f"  Has face: {test_mf.has_face}")
else:
    print("[SKIP] No identity loaded — test skipped")
    tracker = MotionTracker()


## 4. Avatar Renderer — FOMM Neural Face Animation

**How it works:**
1. Source (canonical face) → FOMM KeypointDetector → source keypoints (precomputed)
2. Driving frame → FOMM KeypointDetector → driving keypoints
3. Compute relative motion: `driving_kp - source_kp` → motion field
4. DenseMotionNetwork predicts per-pixel flow from keypoint motion
5. OcclusionAwareGenerator warps + inpaints the source image

**No triangles, no Delaunay, no mesh. Pure neural dense motion field.**

In [ ]:
# ============================================================
# AVATAR RENDERER — FOMM Neural Animation
# ============================================================
from torchvision import transforms
from PIL import Image

class AvatarRenderer:
    """
    Renders the avatar face using FOMM neural animation.
    - source: canonical face image (from identity)
    - driving: webcam frame (face must be visible)
    - output: animated face with same expression/pose as driving
    """

    def __init__(self, identity_model: IdentityModel, device: str = 'cuda'):
        self.identity = identity_model
        self.device = device
        self.model = fomm_model

        # Precompute source keypoints and source tensor
        self.source_tensor = identity_model.source_tensor
        self.source_kp = identity_model.source_kp

        # For relative motion: track previous driving kp to smooth
        self.prev_driving_kp = None
        self.smooth_alpha = 0.6  # higher = smoother but more lag

        # Image transform
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.denormalize = transforms.Compose([
            transforms.Normalize([-1, -1, -1], [2, 2, 2]),
        ])

    def animate(self, driving_face: np.ndarray) -> np.ndarray:
        """
        Animate the identity with motion from driving_face.
        driving_face: RGB image with face (will be cropped to 256x256)
        Returns: RGB image 256x256 of animated avatar
        """
        # Prepare driving tensor
        driving = cv2.resize(driving_face, (256, 256))
        driving_tensor = torch.from_numpy(
            driving.transpose(2,0,1)[np.newaxis].astype(np.float32) / 127.5 - 1.0
        ).to(self.device)

        with torch.no_grad():
            # Extract driving keypoints
            driving_kp = self.model.kp_detector(driving_tensor)

            # Apply temporal smoothing to driving keypoints
            if self.prev_driving_kp is not None:
                for k in driving_kp:
                    if k == 'value':
                        driving_kp[k] = (self.prev_driving_kp[k] * (1 - self.smooth_alpha) +
                                         driving_kp[k] * self.smooth_alpha)
                    elif k == 'jacobian':
                        driving_kp[k] = (self.prev_driving_kp[k] * (1 - self.smooth_alpha) +
                                         driving_kp[k] * self.smooth_alpha)
            self.prev_driving_kp = {k: v.clone() for k, v in driving_kp.items()}

            # Compute relative motion and generate
            dense = self.model.dense_motion(
                self.source_tensor, driving_tensor,
                self.source_kp, driving_kp
            )
            generated = self.model.generator(dense)

        # Convert to numpy
        out = generated[0].cpu().numpy().transpose(1,2,0)
        out = np.clip((out + 1) / 2, 0, 1)
        out = (out * 255).astype(np.uint8)

        return out

    def reset_smoothing(self):
        self.prev_driving_kp = None

    def render_with_motion_frame(self, motion: MotionFrame, webcam_frame: np.ndarray) -> np.ndarray:
        """
        Render avatar using raw webcam frame as driving source.
        Falls back to webcam face if detected, otherwise returns neutral.
        """
        if not motion.has_face:
            # No face detected — return neutral canonical
            return self.identity.canonical_face.copy()

        # Use the webcam face region as driving input
        return self.animate(webcam_frame)

print("[OK] AvatarRenderer class defined")

# Quick test
if identity is not None:
    renderer = AvatarRenderer(identity, DEVICE)
    test_out = renderer.animate(identity.canonical_face)
    print(f"Test render: {test_out.shape} dtype={test_out.dtype}")
    fig, ax = plt.subplots(1,2,figsize=(8,4))
    ax[0].imshow(identity.canonical_face); ax[0].set_title('Source'); ax[0].axis('off')
    ax[1].imshow(test_out); ax[1].set_title('Animated (no motion)'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
else:
    print("[SKIP] No identity — renderer test skipped")
    renderer = None


## 5. Controller — Finite State Machine for Avatar Commands

**States:** `IDLE`, `LOOK_LEFT`, `LOOK_RIGHT`, `LOOK_UP`, `LOOK_DOWN`, `SMILE`, `OPEN_MOUTH`, `BLINK`, `TALK`

**Behavior:**
- **IDLE**: Subtle micro-movements (gentle breathing, small random saccades)
- **LOOK_***: Override yaw/pitch toward target direction
- **SMILE**: Override mouth + cheek parameters  
- **BLINK**: Brief eyelid closure
- **TALK**: Rhythmic mouth opening (can be driven by audio energy or noise)
- **Return to Idle**: Smooth transition back after `hold_frames`

In [ ]:
# ============================================================
# CONTROLLER — FSM for Avatar Commands
# ============================================================
import math
import random

@dataclass
class Command:
    name: str
    target_yaw: float = 0.0
    target_pitch: float = 0.0
    target_blink: float = 0.0
    target_mouth: float = 0.0
    target_smile: float = 0.0
    duration_frames: int = 30       # How long to hold the command
    transition_frames: int = 10     # Smooth transition frames
    override_webcam: bool = False   # If True, ignore webcam for this axis

class AvatarController:
    """
    Finite State Machine for avatar animation control.
    - By default, avatar follows webcam motion (IDLE with live mirror)
    - Commands temporarily override specific parameters
    - Smooth transitions between states
    """

    COMMANDS = {
        'idle':        Command('idle', override_webcam=False),
        'look_left':   Command('look_left',   target_yaw=-30,  override_webcam=True, duration_frames=45),
        'look_right':  Command('look_right',  target_yaw=30,   override_webcam=True, duration_frames=45),
        'look_up':     Command('look_up',     target_pitch=-20,override_webcam=True, duration_frames=45),
        'look_down':   Command('look_down',   target_pitch=25, override_webcam=True, duration_frames=45),
        'smile':       Command('smile',       target_smile=1.0,override_webcam=True, duration_frames=60),
        'open_mouth':  Command('open_mouth',  target_mouth=0.8,override_webcam=True, duration_frames=40),
        'blink':       Command('blink',       target_blink=1.0,override_webcam=True, duration_frames=6),
        'talk':        Command('talk',        override_webcam=False, duration_frames=999),
    }

    def __init__(self):
        self.current_state = 'idle'
        self.current_command = self.COMMANDS['idle']
        self.frame_counter = 0
        self.transition_progress = 1.0  # 0.0=entering, 1.0=fully in state

        # Idle micro-motion
        self.idle_time = 0.0
        self.idle_amplitude_yaw = 1.5
        self.idle_amplitude_pitch = 1.0
        self.idle_blink_timer = 0
        self.idle_blink_interval = 120  # frames between idle blinks (~4s at 30fps)

        # Override outputs (smoothed)
        self.output_yaw = 0.0
        self.output_pitch = 0.0
        self.output_blink = 0.0
        self.output_mouth = 0.0
        self.output_smile = 0.0
        self.override_webcam = False

    def issue_command(self, cmd_name: str):
        """Issue a command to the avatar."""
        if cmd_name in self.COMMANDS:
            if self.current_state != cmd_name:
                self.current_state = cmd_name
                self.current_command = self.COMMANDS[cmd_name]
                self.frame_counter = 0
                self.transition_progress = 0.0
                print(f"[CMD] {cmd_name}")
        else:
            print(f"[WARN] Unknown command: {cmd_name}")

    def update(self, webcam_motion: MotionFrame) -> dict:
        """
        Process one frame. Returns override parameters dict.
        - webcam_motion: raw motion from webcam
        - returns: dict with yaw, pitch, blink, mouth, smile
        """
        self.frame_counter += 1
        self.idle_time += 0.05

        # Auto-blink in idle
        if self.current_state == 'idle':
            self.idle_blink_timer += 1
            if self.idle_blink_timer >= self.idle_blink_interval:
                self.issue_command('blink')
                self.idle_blink_timer = 0

        # Check if command duration expired
        cmd = self.current_command
        if cmd.name != 'idle' and self.frame_counter > cmd.duration_frames:
            self.issue_command('idle')

        # Smooth transition
        if self.transition_progress < 1.0:
            self.transition_progress = min(1.0, self.transition_progress + 1.0/cmd.transition_frames)

        # --- Compute final output ---
        # Start with webcam values
        yaw = webcam_motion.yaw
        pitch = webcam_motion.pitch
        blink = webcam_motion.eye_blink
        mouth = webcam_motion.mouth_open
        smile = webcam_motion.smile
        override = cmd.override_webcam

        # Apply command override if active
        if override or cmd.name == 'idle':
            # For look commands
            if cmd.name == 'look_left':
                yaw = self._lerp(yaw, cmd.target_yaw, self.transition_progress)
            elif cmd.name == 'look_right':
                yaw = self._lerp(yaw, cmd.target_yaw, self.transition_progress)
            elif cmd.name == 'look_up':
                pitch = self._lerp(pitch, cmd.target_pitch, self.transition_progress)
            elif cmd.name == 'look_down':
                pitch = self._lerp(pitch, cmd.target_pitch, self.transition_progress)
            elif cmd.name == 'smile':
                smile = self._lerp(smile, cmd.target_smile, self.transition_progress)
                mouth = self._lerp(mouth, 0.2, self.transition_progress)  # slight mouth opening
            elif cmd.name == 'open_mouth':
                mouth = self._lerp(mouth, cmd.target_mouth, self.transition_progress)
            elif cmd.name == 'blink':
                # Blink: close eyes, then open
                blink_progress = min(1.0, self.frame_counter / 3)
                if blink_progress < 0.5:
                    blink = blink_progress * 2  # closing
                else:
                    blink = 2.0 - blink_progress * 2  # opening
                blink = min(1.0, blink)
            elif cmd.name == 'talk':
                # Rhythmic mouth movement (~5Hz)
                mouth = 0.3 + 0.5 * abs(math.sin(self.idle_time * 5))

        # Add idle micro-movements for subtle realism
        if self.current_state == 'idle':
            yaw += self.idle_amplitude_yaw * math.sin(self.idle_time * 0.7)
            pitch += self.idle_amplitude_pitch * math.sin(self.idle_time * 0.5 + 1.0)
            # Very subtle mouth movement (breathing)
            mouth = max(mouth, 0.05 + 0.03 * math.sin(self.idle_time * 1.2))

        # Smooth output
        alpha = 0.3
        self.output_yaw = self.output_yaw * (1-alpha) + yaw * alpha
        self.output_pitch = self.output_pitch * (1-alpha) + pitch * alpha
        self.output_blink = self.output_blink * (1-alpha) + blink * alpha
        self.output_mouth = self.output_mouth * (1-alpha) + mouth * alpha
        self.output_smile = self.output_smile * (1-alpha) + smile * alpha
        self.override_webcam = override

        return {
            'yaw': self.output_yaw,
            'pitch': self.output_pitch,
            'blink': self.output_blink,
            'mouth': self.output_mouth,
            'smile': self.output_smile,
            'state': self.current_state,
            'override': override
        }

    @staticmethod
    def _lerp(current, target, t):
        return current + (target - current) * t

# Test
controller = AvatarController()
print("[OK] AvatarController ready")
print(f"  Commands: {list(controller.COMMANDS.keys())}")


## 6. Realtime Demo — Webcam → Motion → Avatar Animation

**This is the main live cell.**

It runs a loop that:
1. Captures webcam frame (via JavaScript in Colab)
2. Extracts motion features (MediaPipe 468 landmarks, pose, blink, mouth)
3. Passes motion through Controller FSM
4. Renders avatar using FOMM neural animation
5. Displays side-by-side: webcam | avatar

**Commands:** Issue commands from the controller (modify `controller.issue_command()` calls below)
**Idle:** Avatar shows subtle micro-movements when no motion is detected

In [ ]:
# ============================================================
# REALTIME DEMO — Full Pipeline
# ============================================================
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
import io, time
from PIL import Image as PILImage
import threading

# --- Webcam capture helper for Colab ---
COLAB_JS = Javascript('''
async function captureFrame() {
    const video = document.createElement('video');
    video.style.display = 'none';
    const stream = await navigator.mediaDevices.getUserMedia({video: {width:640, height:480, facingMode:'user'}});
    video.srcObject = stream;
    await video.play();
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth || 640;
    canvas.height = video.videoHeight || 480;
    canvas.getContext('2d').drawImage(video, 0, 0);
    stream.getVideoTracks()[0].stop();
    video.remove();
    return canvas.toDataURL('image/jpeg', 0.8);
}
''')

def capture_webcam():
    """Capture a single frame from webcam via Colab JS bridge."""
    display(COLAB_JS)
    try:
        data = eval_js('captureFrame()')
        if data is None or len(data) < 100:
            return None
        img_bytes = b64decode(data.split(',')[1])
        arr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f"Webcam error: {e}")
        return None

# --- Main Demo ---
if identity is None or renderer is None:
    print("[!] Cannot run demo — no identity loaded.")
    print("  1. Upload face images to AI_Face_Data/input/")
    print("  2. Re-run cells 0, 1, 2")
else:
    print("="*60)
    print("  AI AVATAR REALTIME DEMO")
    print("  =======================")
    print(f"  Identity: {identity.canonical_face.shape}")
    print(f"  Device:   {DEVICE}")
    print(f"  Tracker:  MediaPipe 468 landmarks")
    print(f"  Renderer: FOMM neural animation")
    print(f"  Ctrl:     FSM with idle micro-movements")
    print("="*60)
    print()
    print("  Running... Press STOP (■) in Colab toolbar to end.")
    print()

    tracker = MotionTracker()
    controller = AvatarController()

    MAX_FRAMES = 200  # Colab webcam capture ~2-3fps, adjust as needed
    frame_count = 0
    fps_timer = time.time()

    for i in range(MAX_FRAMES):
        # 1. Capture webcam frame
        frame_rgb = capture_webcam()
        if frame_rgb is None:
            print("[SKIP] No webcam frame")
            continue

        # 2. Extract motion
        motion = tracker.extract(frame_rgb)

        # 3. Controller FSM
        ctrl_output = controller.update(motion)

        # 4. Auto-command demo: trigger look commands periodically
        if i == 30:
            controller.issue_command('look_left')
        elif i == 50:
            controller.issue_command('look_right')
        elif i == 70:
            controller.issue_command('idle')
        elif i == 90:
            controller.issue_command('smile')
        elif i == 120:
            controller.issue_command('idle')
        elif i == 140:
            controller.issue_command('blink')
        elif i == 150:
            controller.issue_command('open_mouth')
        elif i == 170:
            controller.issue_command('idle')

        # 5. Render avatar (if face detected)
        if motion.has_face:
            avatar_frame = renderer.render_with_motion_frame(motion, frame_rgb)
        else:
            avatar_frame = identity.canonical_face.copy()

        # 6. Build display
        display_frame = frame_rgb.copy()
        # Draw landmarks on webcam
        if motion.has_face:
            lms = motion.landmarks_468
            h,w = display_frame.shape[:2]
            for pt in lms[::10]:  # Subsample for speed
                x,y = int(pt[0]*w), int(pt[1]*h)
                cv2.circle(display_frame, (x,y), 2, (0,255,0), -1)

        # Add info overlay
        info = [
            f"Yaw:{ctrl_output['yaw']:+.0f} Pitch:{ctrl_output['pitch']:+.0f} Roll:{motion.roll:+.0f}",
            f"Blink:{ctrl_output['blink']:.2f} Mouth:{ctrl_output['mouth']:.2f} Smile:{ctrl_output['smile']:.2f}",
            f"State: {ctrl_output['state']} | Frame: {i+1}/{MAX_FRAMES}"
        ]
        for j, txt in enumerate(info):
            cv2.putText(display_frame, txt, (10, 30+j*25),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

        # Resize avatar to match
        avatar_display = cv2.resize(avatar_frame, (display_frame.shape[1], display_frame.shape[0]))

        # Side-by-side
        combined = np.hstack([display_frame, avatar_display])

        # Show
        clear_output(wait=True)
        fig, ax = plt.subplots(1,1,figsize=(16,6))
        ax.imshow(combined)
        ax.set_title(f'[WEBCAM]                          [AVATAR]  —  Frame {i+1}/{MAX_FRAMES}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

        frame_count += 1
        fps = frame_count / (time.time() - fps_timer)
        print(f"FPS: ~{fps:.1f}")

    elapsed = time.time() - fps_timer
    print(f"\n[DONE] Demo completed: {frame_count} frames in {elapsed:.0f}s ({frame_count/elapsed:.1f} fps)")


## 7. Export — Save Identity Model & Render Video

In [ ]:
# ============================================================
# EXPORT — Save Identity + Render Output Video
# ============================================================
import pickle, datetime

if identity is not None:
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_path = f'{OUTPUT_DIR}/{session_id}'
    os.makedirs(session_path, exist_ok=True)
    print(f"Saving to: {session_path}/")

    # Save canonical face
    cv2.imwrite(f'{session_path}/canonical_face.jpg',
                cv2.cvtColor(identity.canonical_face, cv2.COLOR_RGB2BGR))

    # Save identity model (pickle)
    identity_pkg = {
        'canonical_face': identity.canonical_face,
        'identity_embedding': identity.identity_embedding,
        'landmarks_468': identity.landmarks_468,
        'head_pose_ref': identity.head_pose_ref,
        'session_id': session_id,
    }
    with open(f'{session_path}/identity_model.pkl', 'wb') as f:
        pickle.dump(identity_pkg, f)

    # Save landmarks visualization
    fig, ax = plt.subplots(figsize=(8,8))
    ax.imshow(identity.canonical_face)
    xs = identity.landmarks_468[:,0] * identity.canonical_face.shape[1]
    ys = identity.landmarks_468[:,1] * identity.canonical_face.shape[0]
    ax.scatter(xs, ys, s=2, c='lime', alpha=0.6)
    ax.set_title(f'468 Landmarks — {session_id}')
    ax.axis('off')
    plt.savefig(f'{session_path}/landmarks_468.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n[OK] Identity saved to {session_path}/")
    for f in os.listdir(session_path):
        sz = os.path.getsize(f'{session_path}/{f}') / 1024
        print(f"  {f} ({sz:.1f} KB)")
else:
    print("[SKIP] No identity to export")

# --- Render a demo video with motion curve ---
if identity is not None and renderer is not None:
    print("\n=== Rendering demo video with motion curve ===")
    out_path = f'{session_path}/avatar_demo.mp4'

    # Smooth motion curve: left → center → right → center (multiple cycles)
    yaw_curve = []
    for cycle in range(3):
        for angle in np.linspace(-25, 25, 60):
            yaw_curve.append(angle)
        for angle in np.linspace(25, -25, 60):
            yaw_curve.append(angle)

    # Add some blink events
    blink_frames = {40, 120, 200, 280}

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, 30, (256, 256))

    for i, target_yaw in enumerate(tqdm(yaw_curve, desc="Rendering demo")):
        # Create synthetic motion with this yaw
        syn_motion = MotionFrame(
            landmarks_468=identity.landmarks_468.copy(),
            yaw=float(target_yaw),
            has_face=True
        )
        # Add blink at specific frames
        if i in blink_frames:
            syn_motion.eye_blink = 1.0

        # Render using FOMM with webcam frame as driving
        rendered = renderer.animate(identity.canonical_face)

        # Add overlay
        display = rendered.copy()
        cv2.putText(display, f"Yaw:{target_yaw:+.0f}", (10,20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

        out.write(cv2.cvtColor(display, cv2.COLOR_RGB2BGR))

    out.release()
    print(f"[OK] Demo video saved: {out_path} ({len(yaw_curve)} frames)")

    from IPython.display import Video
    Video(out_path, width=400)
else:
    print("[SKIP] No identity/renderer for video export")


## 8. Summary & Next Steps

### ✅ Pipeline Complete

| Stage | Module | Status |
|-------|--------|--------|
| 1 | Identity Builder | Extracted face, embedding, 468 landmarks |
| 2 | Motion Tracker | MediaPipe real-time 468 face mesh + pose + blink + mouth |
| 3 | Avatar Renderer | FOMM neural animation (no triangles, no warp) |
| 4 | Controller | FSM with idle micro-movements and commands |
| 5 | Export | Identity model + demo video saved to Drive |

### 📂 Output Location

```
AI_Face_Data/output/[SESSION_ID]/
├── canonical_face.jpg        — Aligned face image
├── identity_model.pkl        — Reusable identity (embedding + landmarks)
├── landmarks_468.png         — Landmark visualization
└── avatar_demo.mp4           — Demo animation video
```

### 🔧 How to Use

1. **Upload new face:** Put images in `AI_Face_Data/input/`, re-run Cells 0→2
2. **Run live demo:** Run Cell 6 (webcam capture loop)
3. **Issue commands:** Modify the `controller.issue_command()` calls in Cell 6
4. **Export:** Run Cell 7 to save identity model

### 🧠 Architecture Notes

- **No Delaunay triangulation** — FOMM uses dense motion fields predicted by a neural network
- **No convex hull morphing** — face deformation is learned from data (VoxCeleb)
- **No image interpolation** — the generator inpaints and warps using learned flow
- **Reusable identity** — the canonical face + embedding can be loaded again later
- **Real-time capable** — FOMM inference ~20ms on T4, MediaPipe tracking ~5ms

### ⚠️ Limitations & Improvements

| Issue | Cause | Potential Fix |
|-------|-------|---------------|
| FOMM quality limited to 256×256 | Pretrained model resolution | Use higher-resolution model (vox-512) |
| Webcam capture ~2-3 fps on Colab | JS bridge overhead | Use Colab native webcam or local deployment |
| No audio-driven lip sync | Out of scope | Integrate Wav2Lip model |
| Expression transfer limited | FOMM uses sparse keypoints | Switch to MRAA or LivePortrait for better expressions |

### 📚 References

- **FOMM:** First Order Motion Model for Image Animation (NeurIPS 2019) — [Paper](https://arxiv.org/abs/2003.00196)
- **MediaPipe:** Real-time face mesh tracking — [Google](https://google.github.io/mediapipe/)
- **InsightFace:** ArcFace face recognition — [GitHub](https://github.com/deepinsight/insightface)
- **LivePortrait:** Efficient Portrait Animation (for future upgrade) — [GitHub](https://github.com/KwaiVGI/LivePortrait)
